In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv
/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e6/train.csv
/kaggle/input/competitions/playground-series-s6e6/test.csv


## Modules

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler

import lightgbm as lgb
import xgboost as xgb

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from tqdm import tqdm
import gc
import os
import random

def seed_everything(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)

print("All imports are successful")

All imports are successful


## data Loading

In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')

original = pd.read_csv('/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv')

print("Train Columns:    ", train.columns.tolist())
print("Original Columns: ", original.columns.tolist())

Train Columns:     ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'class']
Original Columns:  ['obj_ID', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'run_ID', 'rerun_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'class', 'redshift', 'plate', 'MJD', 'fiber_ID']


In [4]:
train.head(10)

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY
5,5,250.727869,31.756548,20.926469,19.693480,18.902361,19.247572,18.508241,0.076299,G/K,Blue_Cloud,STAR
6,6,0.752529,-2.936677,22.829195,22.686143,20.583886,19.781338,19.410491,0.575080,M,Red_Sequence,GALAXY
7,7,235.611325,39.626517,22.511467,21.480306,21.765645,21.508658,21.333476,2.159269,O/B,Blue_Cloud,QSO
8,8,355.359230,2.182312,20.396550,20.064767,19.892257,19.836272,19.860081,0.900087,A/F,Blue_Cloud,QSO
9,9,254.980080,38.743449,18.839137,17.997845,18.458894,18.229552,19.202247,0.114302,O/B,Blue_Cloud,STAR


In [5]:
test['spectral_type'].unique()

array(['G/K', 'M', 'O/B', 'A/F'], dtype=object)

In [ ]:
test.isnull().sum()

## Feature Engineering

In [6]:
scaler_params = {}

def feature_engineering(df, fit=True, scaler=None, ohe_categories=None):
    df = df.copy()

    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    # -------Handling thee -9999 sentinal values -------------
    num_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
    for col in num_cols:
        df[col] = df[col].replace(-9999, np.nan)
        if fit:
            fill_val = df[col].median()
            scaler_params[f'{col}_median_fill'] = fill_val
        df[col] = df[col].fillna(scaler_params[f'{col}_median_fill'])

    # --------- galaxy_populatiion: Binary-------------
    
    if fit:
        gp_unique = sorted(df['galaxy_population'].dropna().unique())
        scaler_params['gp_map'] = {val: i for i, val in enumerate(gp_unique)}
        print("galaxy_population_mapping: ", scaler_params['gp_map'])

    unmapped = df['galaxy_population'].map(scaler_params['gp_map']).isnull().sum()
    if unmapped > 0:
        print(f"WARNING: {unmapped} unseen galaxy_population values found")

    df['galaxy_population'] = df['galaxy_population'].map(
        scaler_params['gp_map']
    ).astype(int)

    # ---Spectral type--- OHE--------
    if fit:
        scaler_params['spectral_dummies'] = sorted(
            df['spectral_type'].dropna().unique()
        )
    for cat in scaler_params['spectral_dummies']:
        clean_name = f"spectral_{cat.replace('/', '_')}"
        df[clean_name] = (df['spectral_type'] == cat).astype(int)
    df = df.drop('spectral_type', axis=1)

    # ------------ Color Pairs ----------------
    df['color_u_g'] = df['u'] - df['g']   # UV slope
    df['color_g_r'] = df['g'] - df['r']   # visible color, best discriminator
    df['color_r_i'] = df['r'] - df['i']   # red region
    df['color_i_z'] = df['i'] - df['z']   # near-IR slope
    df['color_u_r'] = df['u'] - df['r']   # broadband, galaxy vs QSO
    df['color_u_z'] = df['u'] - df['z']   # widest baseline, QSO identifier
    df['color_g_z'] = df['g'] - df['z']   # another wide baseline

    # Red shift features
    df['redshift_log'] = np.log1p(np.abs(df['redshift']))
    df['redshift_sq'] = df['redshift'] ** 2

    df['rs_x_gr']  = df['redshift'] * df['color_g_r']
    df['rs_x_ug']  = df['redshift'] * df['color_u_g']
    df['rs_x_ri']  = df['redshift'] * df['color_r_i']
    df['rs_x_uz']  = df['redshift'] * df['color_u_z']

    # ── 7. Magnitude features ─────────────────────────────────────────────────
    mag_cols = ['u', 'g', 'r', 'i', 'z']
    df['mean_mag']   = df[mag_cols].mean(axis=1)
    df['mag_range']  = df[mag_cols].max(axis=1) - df[mag_cols].min(axis=1)
    df['r_dev']      = df['r'] - df['mean_mag']   # r-band deviation
    df['u_dev']      = df['u'] - df['mean_mag']   # u-band UV excess flag

    # ── 8. Redshift bin (meaningful shortcut for tree models) ─────────────────
    df['redshift_bin'] = pd.cut(
        df['redshift'],
        bins=[-np.inf, 0.002, 1.0, np.inf],
        labels=[0, 1, 2]          # 0=STAR zone, 1=GALAXY zone, 2=QSO zone
    ).astype(int)

    # ── 8. Redshift bin (meaningful shortcut for tree models) ─────────────────
    df['redshift_bin'] = pd.cut(
        df['redshift'],
        bins=[-np.inf, 0.002, 1.0, np.inf],
        labels=[0, 1, 2]          # 0=STAR zone, 1=GALAXY zone, 2=QSO zone
    ).astype(int)

    # ── 9. Spatial / trig features ────────────────────────────────────────────
    df['alpha_rad']   = np.deg2rad(df['alpha'])
    df['delta_rad']   = np.deg2rad(df['delta'])
    df['sin_alpha']   = np.sin(df['alpha_rad'])
    df['cos_alpha']   = np.cos(df['alpha_rad'])
    df['sin_delta']   = np.sin(df['delta_rad'])
    df['cos_delta']   = np.cos(df['delta_rad'])
    # galactic latitude proxy — stars cluster near b=0
    df['galactic_lat'] = np.degrees(
        np.arcsin(
            np.sin(df['delta_rad']) * np.sin(np.deg2rad(62.6)) -
            np.cos(df['delta_rad']) * np.cos(df['alpha_rad']) * np.cos(np.deg2rad(62.6))
        )
    )
    # drop intermediate radian cols — not needed as features
    df = df.drop(['alpha_rad', 'delta_rad'], axis=1)

    
    # ── 10. Robust Scaling ────────────────────────────────────────────────────
    cols_to_scale = [
        # original
        'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
        # engineered numerical
        'color_u_g', 'color_g_r', 'color_r_i', 'color_i_z',
        'color_u_r', 'color_u_z', 'color_g_z',
        'redshift_log', 'redshift_sq',
        'rs_x_gr', 'rs_x_ug', 'rs_x_ri', 'rs_x_uz',
        'mean_mag', 'mag_range', 'r_dev', 'u_dev',
        'galactic_lat',
        'sin_alpha', 'cos_alpha', 'sin_delta', 'cos_delta',
    ]

    if fit:
        scaler_params['medians'] = df[cols_to_scale].median()
        scaler_params['iqr'] = (df[cols_to_scale].quantile(0.75) - df[cols_to_scale].quantile(0.25))
        scaler_params['iqr'] = scaler_params['iqr'].replace(0,1)

    df[cols_to_scale] = (
        df[cols_to_scale] - scaler_params['medians']
    ) / scaler_params['iqr']

    return df

# -------Apply ---------------------
le = LabelEncoder()
y  = le.fit_transform(train['class'])
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

X      = train.drop('class', axis=1)
X_test = test.copy()

X      = feature_engineering(X,      fit=True)
X_test = feature_engineering(X_test, fit=False)

print("X shape     :", X.shape)
print("X_test shape:", X_test.shape)
print("Nulls in X  :", X.isnull().sum().sum())
print("Features    :", X.columns.tolist())


Class mapping: {'GALAXY': np.int64(0), 'QSO': np.int64(1), 'STAR': np.int64(2)}
galaxy_population_mapping:  {'Blue_Cloud': 0, 'Red_Sequence': 1}
X shape     : (577347, 36)
X_test shape: (247435, 36)
Nulls in X  : 0
Features    : ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'galaxy_population', 'spectral_A_F', 'spectral_G_K', 'spectral_M', 'spectral_O_B', 'color_u_g', 'color_g_r', 'color_r_i', 'color_i_z', 'color_u_r', 'color_u_z', 'color_g_z', 'redshift_log', 'redshift_sq', 'rs_x_gr', 'rs_x_ug', 'rs_x_ri', 'rs_x_uz', 'mean_mag', 'mag_range', 'r_dev', 'u_dev', 'redshift_bin', 'sin_alpha', 'cos_alpha', 'sin_delta', 'cos_delta', 'galactic_lat']


In [7]:
# Should be no nulls
print("Nulls in X:", X.isnull().sum().sum())

# Check spectral_type OHE worked
spectral_cols = [c for c in X.columns if c.startswith('spectral_')]
print("Spectral OHE cols:", spectral_cols)

# Check redshift_bin distribution
print("Redshift bin counts:\n", 
      pd.Series(X['redshift_bin']).value_counts().sort_index())

Nulls in X: 0
Spectral OHE cols: ['spectral_A_F', 'spectral_G_K', 'spectral_M', 'spectral_O_B']
Redshift bin counts:
 redshift_bin
0     14120
1    444227
2    119000
Name: count, dtype: int64


## LGBM Baseline

In [ ]:
FOLDS = 5
SEED = 42

lgbm_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'verbosity': -1,
    'random_state': SEED,

    #'device': 'cuda',

    'n_estimators': 3000,
    'learning_rate': 0.05,
    'max_depth': 7,
    'num_leaves': 127,
    'min_child_samples': 20,

    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'min_split_gain': 0.01,

    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,

    'class_weight': 'balanced',
}

In [ ]:
## KFold Training Loop
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros((len(X), 3))
test_preds = np.zeros((len(X_test), 3))
scores= []
importances = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'='*40}")
    print(f"  Fold {fold} / {FOLDS}")
    print(f"{'='*40}")

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**lgbm_params)

    model.fit(
        X_tr, y_tr,
        eval_set = [(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds = 100, verbose=True),
            lgb.log_evaluation(period=200),
        ]
    )

    # ── OOF ───────────────────────────────────────────────────────────────
    oof_preds[val_idx] = model.predict_proba(X_val)

    # ── Test ───────────────────────────────────────────────────────────────
    test_preds += model.predict_proba(X_test) / FOLDS

    # ── Score ───────────────────────────────────────────────────────────────
    fold_score = balanced_accuracy_score(
        y_val, oof_preds[val_idx].argmax(axis=1)
    )
    scores.append(fold_score)
    print(f"\n  Fold {fold} Score: {fold_score:.5f}")

    # ── Importance ───────────────────────────────────────────────────────────────
    importances.append(
        pd.Series(model.feature_importances_, index = X.columns)
    )
    
    del X_tr, X_val, y_tr, y_val, model
    gc.collect()

# ── Final Results ─────────────────────────────────────────────────────────────
oof_score = balanced_accuracy_score(y, oof_preds.argmax(axis=1))
print(f"\n{'='*40}")
print(f"  OOF Score:       {oof_score:.5f}")
print(f"  Per fold:        {[round(s, 5) for s in scores]}")
print(f"  Std deviation:   {np.std(scores):.5f}")
print(f"{'='*40}")

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
mean_importance = pd.DataFrame(importances).mean().sort_values(ascending=False)

plt.figure(figsize=(10, 8))
mean_importance.head(25).plot(kind='barh')
plt.title('Top 25 Feature Importances (avg across 5 folds)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(mean_importance.head(10))

In [ ]:
# ── Submission ────────────────────────────────────────────────────────────────
sub = pd.DataFrame({
    'id':    test['id'],
    'class': le.inverse_transform(test_preds.argmax(axis=1))
})
sub.to_csv('submission_lgbm_baseline.csv', index=False)
print("Submission saved!")
print(sub['class'].value_counts())

## XGB Baseline

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

xgb_params = {
    'objective':        'multi:softprob',
    'num_class':        3,
    'eval_metric':      'mlogloss',
    'random_state':     SEED,
    'verbosity':        0,

    # GPU
    'device':           'cuda',
    'tree_method':      'hist',

    # Tree structure
    'n_estimators':     3000,
    'learning_rate':    0.05,
    'max_depth':        7,
    'min_child_weight': 20,      # smin_child_samples in LGBM

    # Regularization
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,

    # Sampling
    'subsample':        0.8,
    'colsample_bytree': 0.8,

    # Early stopping
    'early_stopping_rounds': 100,
}



In [ ]:
# ── XGB KFold Loop ────────────────────────────────────────────────────────────
xgb_oof_preds  = np.zeros((len(X), 3))
xgb_test_preds = np.zeros((len(X_test), 3))
xgb_scores     = []
xgb_importances = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'='*40}")
    print(f"  Fold {fold} / {FOLDS}")
    print(f"{'='*40}")

    X_tr,  X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr,  y_val = y[tr_idx],      y[val_idx]

    # XGBoost doesn't have class_weight param for multiclass
    # so we pass sample weights directly to fit()
    sample_weights = compute_sample_weight('balanced', y_tr)

    model = xgb.XGBClassifier(**xgb_params)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        sample_weight=sample_weights,
        verbose=200,
    )

    # ── OOF ───────────────────────────────────────────────────────────────
    xgb_oof_preds[val_idx] = model.predict_proba(X_val)

    # ── Test ──────────────────────────────────────────────────────────────
    xgb_test_preds += model.predict_proba(X_test) / FOLDS

    # ── Score ─────────────────────────────────────────────────────────────
    fold_score = balanced_accuracy_score(
        y_val, xgb_oof_preds[val_idx].argmax(axis=1)
    )
    xgb_scores.append(fold_score)
    print(f"\n  Fold {fold} Score: {fold_score:.5f}")

    # ── Importance ────────────────────────────────────────────────────────
    xgb_importances.append(
        pd.Series(model.feature_importances_, index=X.columns)
    )

    del X_tr, X_val, y_tr, y_val, model, sample_weights
    gc.collect()

# ── Final Results ─────────────────────────────────────────────────────────────
xgb_oof_score = balanced_accuracy_score(y, xgb_oof_preds.argmax(axis=1))
print(f"\n{'='*40}")
print(f"  XGB OOF Score:   {xgb_oof_score:.5f}")
print(f"  Per fold:        {[round(s, 5) for s in xgb_scores]}")
print(f"  Std deviation:   {np.std(xgb_scores):.5f}")
print(f"{'='*40}")

In [ ]:
# ── XGB Feature Importance ────────────────────────────────────────────────────
xgb_mean_importance = pd.DataFrame(xgb_importances).mean().sort_values(ascending=False)

plt.figure(figsize=(10, 8))
xgb_mean_importance.head(25).plot(kind='barh')
plt.title('XGB Top 25 Feature Importances (avg across folds)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nXGB Top 10 features:")
print(xgb_mean_importance.head(10))

In [ ]:
# ── XGB Submission ────────────────────────────────────────────────────────────
xgb_sub = pd.DataFrame({
    'id':    test['id'],
    'class': le.inverse_transform(xgb_test_preds.argmax(axis=1))
})
xgb_sub.to_csv('submission_xgb_baseline.csv', index=False)
print("XGB Submission saved!")
print(xgb_sub['class'].value_counts())

## Tuning with Optuna

### LGBM

In [ ]:
'''# ── Stratified subsample for tuning ──────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_tune, _, y_tune, _ = train_test_split(
    X, y,
    train_size=100_000,     # 100k rows — fast enough per trial
    stratify=y,             # maintains class proportions
    random_state=42
)
print(f"Tune set shape:      {X_tune.shape}")
print(f"Class distribution:\n{pd.Series(y_tune).value_counts()}")'''

In [8]:
# ── Optuna LGBM Tuning ────────────────────────────────────────────────────────
def lgbm_objective(trial):
    params = {
        # Fixed — don't tune these
        'objective':     'multiclass',
        'num_class':     3,
        'metric':        'multi_logloss',
        'verbosity':     -1,
        'random_state':  42,
        'class_weight':  'balanced',
        'n_estimators':  2000,
        'subsample_freq': 1,

        # Tunable parameters
        'learning_rate': trial.suggest_float(
            'learning_rate', 0.01, 0.1, log=True
        ),
        'num_leaves': trial.suggest_int(
            'num_leaves', 31, 255
        ),
        'max_depth': trial.suggest_int(
            'max_depth', 4, 10
        ),
        'min_child_samples': trial.suggest_int(
            'min_child_samples', 10, 100
        ),
        'reg_alpha': trial.suggest_float(
            'reg_alpha', 1e-8, 10.0, log=True
        ),
        'reg_lambda': trial.suggest_float(
            'reg_lambda', 1e-8, 10.0, log=True
        ),
        'subsample': trial.suggest_float(
            'subsample', 0.5, 1.0
        ),
        'colsample_bytree': trial.suggest_float(
            'colsample_bytree', 0.5, 1.0
        ),
        'min_split_gain': trial.suggest_float(
            'min_split_gain', 0.0, 1.0
        ),
    }

    # 3-fold instead of 5 for speed — enough to estimate performance
    cv     = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for tr_idx, val_idx in cv.split(X, y):
        X_tr,  X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr,  y_val = y[tr_idx],      y[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(100,  verbose=False),
                lgb.log_evaluation(-1),   # completely silent
            ]
        )

        preds = model.predict_proba(X_val).argmax(axis=1)
        scores.append(balanced_accuracy_score(y_val, preds))

        del X_tr, X_val, y_tr, y_val, model
        gc.collect()

    return np.mean(scores)

In [10]:
sampler = TPESampler(seed=42)
study = optuna.create_study(
    direction = 'maximize',
    sampler=sampler,
    study_name='lgbm_tuning'
)

study.optimize(
    lgbm_objective,
    n_trials=50,
    show_progress_bar=True
)

print(f"\nBest trial:  {study.best_trial.number}")
print(f"Best score:  {study.best_value:.5f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-06-12 05:19:30,296] Trial 7 failed with parameters: {'learning_rate': 0.02273805573563183, 'num_leaves': 94, 'max_depth': 7, 'min_child_samples': 22, 'reg_alpha': 0.16587190283399655, 'reg_lambda': 4.6876566400928895e-08, 'subsample': 0.9934434683002586, 'colsample_bytree': 0.8861223846483287, 'min_split_gain': 0.1987156815341724} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_98/1448250932.py", line 53, in lgbm_objective
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py", lin

KeyboardInterrupt: 

In [ ]:

optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout()
plt.show()

optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout()
plt.show()

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
FOLDS=5
# ── Retrain with Best Params on Full 5-Fold ───────────────────────────────────
best_params = {
    # Fixed
    'objective':      'multiclass',
    'num_class':      3,
    'metric':         'multi_logloss',
    'verbosity':      -1,
    'random_state':   42,
    'class_weight':   'balanced',
    'subsample_freq': 1,
    'n_estimators':   3000,   # higher ceiling since we use best lr now

    # From Optuna
    **study.best_params
}

lgbm_tuned_oof   = np.zeros((len(X), 3))
lgbm_tuned_test  = np.zeros((len(X_test), 3))
lgbm_tuned_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'='*40}")
    print(f"  Tuned LGBM Fold {fold} / {FOLDS}")
    print(f"{'='*40}")

    X_tr,  X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr,  y_val = y[tr_idx],      y[val_idx]

    model = lgb.LGBMClassifier(**best_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(100, verbose=True),
            lgb.log_evaluation(200),
        ]
    )

    lgbm_tuned_oof[val_idx]  = model.predict_proba(X_val)
    lgbm_tuned_test          += model.predict_proba(X_test) / FOLDS

    fold_score = balanced_accuracy_score(
        y_val, lgbm_tuned_oof[val_idx].argmax(axis=1)
    )
    lgbm_tuned_scores.append(fold_score)
    print(f"\n  Fold {fold} Score: {fold_score:.5f}")

    del X_tr, X_val, y_tr, y_val, model
    gc.collect()

lgbm_tuned_oof_score = balanced_accuracy_score(
    y, lgbm_tuned_oof.argmax(axis=1)
)
print(f"\n{'='*40}")
print(f"  Tuned LGBM OOF:  {lgbm_tuned_oof_score:.5f}")
print(f"  Baseline was:    0.96453")
print(f"  Improvement:     {lgbm_tuned_oof_score - 0.96453:+.5f}")
print(f"{'='*40}")

In [ ]:
# ── XGB Submission ────────────────────────────────────────────────────────────
lgbm_tune_sub = pd.DataFrame({
    'id':    test['id'],
    'class': le.inverse_transform(lgbm_tuned_test.argmax(axis=1))
})
lgbm_tune_sub.to_csv('submission_lgbm_tuned.csv', index=False)
print("LGBM tune Submission saved!")
print(lgbm_tune_sub['class'].value_counts())

### XGB

In [ ]:
# ── Optuna XGB Tuning ─────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        # Fixed
        'objective':     'multi:softprob',
        'num_class':     3,
        'eval_metric':   'mlogloss',
        'random_state':  SEED,
        'verbosity':     0,
        'device':        'cuda',
        'tree_method':   'hist',
        'n_estimators':  2000,
        'early_stopping_rounds': 50,

        # Tunable
        'learning_rate': trial.suggest_float(
            'learning_rate', 0.01, 0.1, log=True
        ),
        'max_depth': trial.suggest_int(
            'max_depth', 4, 10
        ),
        'min_child_weight': trial.suggest_int(
            'min_child_weight', 10, 100
        ),
        'reg_alpha': trial.suggest_float(
            'reg_alpha', 1e-8, 10.0, log=True
        ),
        'reg_lambda': trial.suggest_float(
            'reg_lambda', 1e-8, 10.0, log=True
        ),
        'subsample': trial.suggest_float(
            'subsample', 0.5, 1.0
        ),
        'colsample_bytree': trial.suggest_float(
            'colsample_bytree', 0.5, 1.0
        ),
        'gamma': trial.suggest_float(
            'gamma', 0.0, 1.0          # min loss reduction to split — XGB equivalent of min_split_gain
        ),
        'grow_policy': trial.suggest_categorical(
            'grow_policy', ['depthwise', 'lossguide']
        ),
    }

    cv     = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []

    for tr_idx, val_idx in cv.split(X, y):
        X_tr,  X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr,  y_val = y[tr_idx],      y[val_idx]

        sample_weights = compute_sample_weight('balanced', y_tr)

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            sample_weight=sample_weights,
            verbose=False,
        )

        preds = model.predict_proba(X_val).argmax(axis=1)
        scores.append(balanced_accuracy_score(y_val, preds))

        del X_tr, X_val, y_tr, y_val, model, sample_weights
        gc.collect()

    return np.mean(scores)

In [ ]:
# ── Run XGB Study ─────────────────────────────────────────────────────────────
xgb_sampler = TPESampler(seed=SEED)
xgb_study   = optuna.create_study(
    direction='maximize',
    sampler=xgb_sampler,
    study_name='xgb_tuning'
)

xgb_study.optimize(
    xgb_objective,
    n_trials=50,
    show_progress_bar=True,
)

print(f"\nBest trial:  {xgb_study.best_trial.number}")
print(f"Best score:  {xgb_study.best_value:.5f}")
print(f"Best params:")
for k, v in xgb_study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Visualize XGB Optuna Results ──────────────────────────────────────────────
optuna.visualization.matplotlib.plot_optimization_history(xgb_study)
plt.tight_layout()
plt.show()

optuna.visualization.matplotlib.plot_param_importances(xgb_study)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
# ── Retrain XGB with Best Params on Full 5-Fold ───────────────────────────────
xgb_best_params = {
    # Fixed
    'objective':             'multi:softprob',
    'num_class':             3,
    'eval_metric':           'mlogloss',
    'random_state':          42,
    'verbosity':             0,
    'device':                'cuda',
    'tree_method':           'hist',
    'n_estimators':          3000,
    'early_stopping_rounds': 100,

    # From Optuna
    'learning_rate':      0.02636,
    'max_depth':          6,
    'min_child_weight':   10,
    'reg_alpha':          4.3809,
    'reg_lambda':         1.7716e-07,
    'subsample':          0.8463,
    'colsample_bytree':   0.6175,
    'gamma':              0.2741,
    'grow_policy':        'depthwise',
}

xgb_tuned_oof    = np.zeros((len(X), 3))
xgb_tuned_test   = np.zeros((len(X_test), 3))
xgb_tuned_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n{'='*40}")
    print(f"  Tuned XGB Fold {fold} / {FOLDS}")
    print(f"{'='*40}")

    X_tr,  X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr,  y_val = y[tr_idx],      y[val_idx]

    sample_weights = compute_sample_weight('balanced', y_tr)

    model = xgb.XGBClassifier(**xgb_best_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        sample_weight=sample_weights,
        verbose=200,
    )

    xgb_tuned_oof[val_idx]  = model.predict_proba(X_val)
    xgb_tuned_test          += model.predict_proba(X_test) / FOLDS

    fold_score = balanced_accuracy_score(
        y_val, xgb_tuned_oof[val_idx].argmax(axis=1)
    )
    xgb_tuned_scores.append(fold_score)
    print(f"\n  Fold {fold} Score: {fold_score:.5f}")

    del X_tr, X_val, y_tr, y_val, model, sample_weights
    gc.collect()

xgb_tuned_oof_score = balanced_accuracy_score(
    y, xgb_tuned_oof.argmax(axis=1)
)
print(f"\n{'='*40}")
print(f"  Tuned XGB OOF:   {xgb_tuned_oof_score:.5f}")
print(f"  Baseline was:    0.96423")
print(f"  Improvement:     {xgb_tuned_oof_score - 0.96423:+.5f}")
print(f"{'='*40}")

In [ ]:
# ── XGB Submission ────────────────────────────────────────────────────────────
xgb_tune_sub = pd.DataFrame({
    'id':    test['id'],
    'class': le.inverse_transform(xgb_tuned_test.argmax(axis=1))
})
xgb_tune_sub.to_csv('submission_xgb_tuned.csv', index=False)
print("XGB tune Submission saved!")
print(xgb_tune_sub['class'].value_counts())

### Ensembling

In [ ]:
# ── Ensemble Both Tuned Models ────────────────────────────────────────────────
# Try different weights and pick best OOF

print("Finding best ensemble weights...\n")
best_ensemble_score = 0
best_weight         = 0

# Try XGB-heavy weights manually
for lgbm_w in [0.1]:
    xgb_w = 1 - lgbm_w
    ensemble = lgbm_w * lgbm_tuned_oof + xgb_w * xgb_tuned_oof
    score = balanced_accuracy_score(y, ensemble.argmax(axis=1))
    print(f"LGBM {lgbm_w:.1f} + XGB {xgb_w:.1f} → {score:.5f}")

    if score > best_ensemble_score:
        best_ensemble_score = score
        best_weight         = lgbm_w

print(f"\n  Best ensemble:   LGBM {best_weight:.1f} + XGB {1-best_weight:.1f}")
print(f"  Best OOF score:  {best_ensemble_score:.5f}")
print(f"  Tuned LGBM was:  {lgbm_tuned_oof_score:.5f}")
print(f"  Tuned XGB was:   {xgb_tuned_oof_score:.5f}")

In [ ]:
# ── Final Ensemble Submission ─────────────────────────────────────────────────
final_test_preds = best_weight * lgbm_tuned_test + (1 - best_weight) * xgb_tuned_test

final_sub = pd.DataFrame({
    'id':    test['id'],
    'class': le.inverse_transform(final_test_preds.argmax(axis=1))
})
final_sub.to_csv('submission_ensemble_final_3.csv', index=False)
print("Final ensemble 3 submission saved!")
print(final_sub['class'].value_counts())